<a href="https://colab.research.google.com/github/run-llama/llama_index/blob/main/docs/examples/agent/gemma4_private_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Private RAG with Local Gemma 4 — Google Drive, Gmail & Local Files

This notebook builds a fully private Retrieval-Augmented Generation (RAG) pipeline that lets you ask questions over your own documents. The LLM (**Gemma 4** via Ollama) and embeddings run 100% locally — your document content never reaches a cloud AI provider.

**Data sources covered:**
1. **Local files** — PDFs, Word docs, Markdown, CSVs, text files
2. **Google Drive** — Docs, Sheets, PDFs stored in your Drive
3. **Gmail** — Search and summarize your emails
4. **Google Keep** — Notes and checklists

**Privacy model:**
- Gemma 4 LLM: 100% local (Ollama)
- Embeddings: 100% local (HuggingFace `bge-small-en-v1.5`)
- Google readers: read-only OAuth; data is indexed in memory on your machine
- Nothing is sent to OpenAI, Anthropic, or any cloud LLM

## Prerequisites

1. **Ollama** running with Gemma 4 pulled:
   ```bash
   ollama serve && ollama pull gemma4:12b
   ```

2. **Google Cloud credentials** (for Drive/Gmail — skip if using local files only):
   - Go to [Google Cloud Console](https://console.cloud.google.com)
   - Create a project → Enable **Google Drive API** and **Gmail API**
   - Create OAuth 2.0 credentials → Download as `credentials.json`
   - Place `credentials.json` in this notebook's directory

In [ ]:
!pip install llama-index-llms-ollama \
             llama-index-embeddings-huggingface \
             llama-index-readers-google \
             llama-index \
             google-auth-oauthlib \
             google-api-python-client

## 1. Set up local LLM and embeddings

In [ ]:
from llama_index.core import Settings
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Local LLM — swap 12b for 2b if RAM is limited
Settings.llm = Ollama(model="gemma4:12b", request_timeout=180.0)

# Local embeddings — downloads once (~130 MB), then runs offline
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

print("LLM and embeddings ready — fully local, no API keys needed.")

## 2. Local file RAG

The simplest starting point — index any files on your machine.
Supports: PDF, DOCX, PPTX, CSV, Markdown, HTML, EPUB, images (with vision), and more.

In [ ]:
import os
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

# Point this at any folder — financial statements, notes, contracts, etc.
LOCAL_DOCS_PATH = "./my_documents"  # <-- change to your folder

os.makedirs(LOCAL_DOCS_PATH, exist_ok=True)

# Example: write a sample document to test with
with open(f"{LOCAL_DOCS_PATH}/sample.md", "w") as f:
    f.write("""
# Personal Finance Summary - June 2026

## Checking (CapFed)
- Balance: $4,230.50
- Recent transactions: Netflix $15.99, Grocery $87.43, Gas $54.00

## Savings (Meritrust)
- Balance: $12,800.00
- Goal: Emergency fund (target $15,000)

## Investment notes
- ETH holding: 0.8 ETH
- BTC holding: 0.02 BTC
- Next review date: July 15, 2026
""")

documents = SimpleDirectoryReader(LOCAL_DOCS_PATH).load_data()
print(f"Loaded {len(documents)} document(s)")

index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine()

print("Index built. Ready to query.")

In [ ]:
response = query_engine.query("What is my current savings balance and how far am I from my goal?")
print(response)

In [ ]:
response = query_engine.query("Summarize my recent spending and identify any recurring subscriptions.")
print(response)

In [ ]:
# Multi-turn chat over your documents
chat_engine = index.as_chat_engine(chat_mode="condense_plus_context")

r1 = chat_engine.chat("What crypto do I hold?")
print("Q1:", r1)

r2 = chat_engine.chat("When should I review those holdings?")
print("Q2:", r2)  # Remembers context from previous turn

### Streaming RAG

In [ ]:
streaming_engine = index.as_query_engine(streaming=True)
response = streaming_engine.query("Give me a financial health summary based on these documents.")
response.print_response_stream()

## 3. Google Drive RAG

Index documents directly from your Google Drive. Requires `credentials.json` from Google Cloud Console.
Only requests **read-only** Drive access (`drive.readonly` scope).

In [ ]:
from llama_index.readers.google import GoogleDriveReader

# First run opens a browser for OAuth consent — token.json is saved for future runs
drive_reader = GoogleDriveReader(
    credentials_path="credentials.json",
    token_path="token.json",
)

# Option A: Load all files from a specific folder
# Find folder_id in the Drive URL: drive.google.com/drive/folders/<folder_id>
DRIVE_FOLDER_ID = "YOUR_FOLDER_ID_HERE"

drive_docs = drive_reader.load_data(folder_id=DRIVE_FOLDER_ID)
print(f"Loaded {len(drive_docs)} document(s) from Google Drive")

In [ ]:
# Option B: Load specific files by ID
# Find file_id in the file URL: docs.google.com/document/d/<file_id>/edit
# drive_docs = drive_reader.load_data(file_ids=["FILE_ID_1", "FILE_ID_2"])

# Option C: Load files matching a query string
# drive_docs = drive_reader.load_data(query_string="name contains 'budget'")

# Build the index from Drive documents
drive_index = VectorStoreIndex.from_documents(drive_docs)
drive_engine = drive_index.as_query_engine()

response = drive_engine.query("Summarize the key points across all my Drive documents.")
print(response)

In [ ]:
# Combine local files + Drive into a single unified index
all_docs = documents + drive_docs
unified_index = VectorStoreIndex.from_documents(all_docs)
unified_engine = unified_index.as_query_engine()

response = unified_engine.query(
    "Based on everything — local files and Drive docs — what are my top financial priorities?"
)
print(response)

## 4. Gmail RAG

Search and summarize your emails locally. Only requests **read-only** Gmail access.
Email content is indexed in memory on your machine — never sent to a cloud LLM.

In [ ]:
from llama_index.readers.google import GmailReader

# Read recent emails — uses OAuth (same credentials.json flow)
gmail_reader = GmailReader(
    query="is:unread",   # Gmail search query — same syntax as Gmail search bar
    max_results=20,
    service=None,        # auto-initialized on first call
    results_per_page=10,
)

email_docs = gmail_reader.load_data()
print(f"Loaded {len(email_docs)} email(s)")

In [ ]:
email_index = VectorStoreIndex.from_documents(email_docs)
email_engine = email_index.as_query_engine()

response = email_engine.query("What are the most important unread emails I need to act on?")
print(response)

In [ ]:
# Search specific topics
finance_reader = GmailReader(
    query="from:noreply@bank OR subject:statement OR subject:invoice",
    max_results=50,
    service=None,
    results_per_page=10,
)
finance_emails = finance_reader.load_data()
print(f"Loaded {len(finance_emails)} finance-related email(s)")

finance_index = VectorStoreIndex.from_documents(finance_emails)
finance_engine = finance_index.as_query_engine()

response = finance_engine.query("Summarize any bank statements, invoices, or payment confirmations.")
print(response)

## 5. Unified "Ask my stuff" agent

Combine all sources — local files, Drive, and Gmail — into one agent with a natural language interface.

In [ ]:
from llama_index.core.tools import QueryEngineTool
from llama_index.core.agent.workflow import ReActAgent

# Wrap each index as a named tool the agent can choose between
local_tool = QueryEngineTool.from_defaults(
    query_engine=query_engine,
    name="local_documents",
    description="Search local files — financial summaries, notes, and personal documents.",
)

drive_tool = QueryEngineTool.from_defaults(
    query_engine=drive_engine,
    name="google_drive",
    description="Search Google Drive documents — contracts, spreadsheets, shared files.",
)

email_tool = QueryEngineTool.from_defaults(
    query_engine=email_engine,
    name="gmail",
    description="Search Gmail inbox — emails, invoices, bank notifications, and receipts.",
)

personal_agent = ReActAgent(
    tools=[local_tool, drive_tool, email_tool],
    llm=Settings.llm,
    system_prompt=(
        "You are a private personal assistant with access to the user's local documents, "
        "Google Drive, and Gmail. Always cite which source you found information in. "
        "Never fabricate data — if you can't find something, say so."
    ),
    verbose=True,
)

print("Personal agent ready — queries stay 100% local.")

In [ ]:
response = await personal_agent.run(
    "Give me a complete picture of my finances — check my local docs, Drive, and any bank emails."
)
print(response)

In [ ]:
response = await personal_agent.run(
    "Are there any emails about upcoming payment deadlines or bills I haven't handled yet?"
)
print(response)

## 6. Persist the index to disk

Save your index so you don't have to re-embed documents every session.

In [ ]:
from llama_index.core import StorageContext, load_index_from_storage

INDEX_PATH = "./my_rag_index"

# Save
unified_index.storage_context.persist(persist_dir=INDEX_PATH)
print(f"Index saved to {INDEX_PATH}")

# Load in a future session (no re-embedding needed)
storage_context = StorageContext.from_defaults(persist_dir=INDEX_PATH)
loaded_index = load_index_from_storage(storage_context)
loaded_engine = loaded_index.as_query_engine()

response = loaded_engine.query("What documents are indexed?")
print(response)

## 7. Keep your index fresh — incremental updates

In [ ]:
from llama_index.core import RefreshSettings

# Add new documents without rebuilding from scratch
new_docs = SimpleDirectoryReader("./new_documents").load_data()

for doc in new_docs:
    unified_index.insert(doc)

# Persist the updated index
unified_index.storage_context.persist(persist_dir=INDEX_PATH)
print(f"Index updated with {len(new_docs)} new document(s)")

## Privacy summary

| Component | Where it runs | Data exposure |
|---|---|---|
| Gemma 4 (LLM) | Your machine via Ollama | None — fully local |
| BGE embeddings | Your machine via HuggingFace | None — fully local |
| Vector index | Your machine (RAM + disk) | None — fully local |
| Google Drive reader | OAuth read-only API call | Only file metadata + content fetched to your machine |
| Gmail reader | OAuth read-only API call | Only matched emails fetched to your machine |

> Your document content is **never sent to OpenAI, Anthropic, or any cloud LLM.** The only external calls are the OAuth-authenticated reads from Google's API to your own account.

## Next steps

- **Step 13** — Add Twilio so the agent can SMS you a daily summary
- **Step 14** — Add Bitly so shared links are automatically shortened
- **Step 25** — Deploy a Vercel web UI so you can query from your phone
- **Step 31** — Add LlamaIndex instrumentation to trace every RAG call